# 08 · 年齢だけのモデルは較正されているか

画像モデルの主要な問題は「AUC は保たれるが、較正が崩れる」ことだった（分布外 ECE 0.104、LOSO-B 0.162）。
一方、画像を使わない年齢 1 変数のロジスティック回帰は、分布外の AUC で画像モデルを上回った（0.889 対 0.860、
`results/age_prior.json` の `auc_table`）。ただしその比較は **AUC だけ**で、年齢モデルの較正は測っていなかった。

ここでは 3 つのスコアを、同じ 9 ユニット（分布外 4・分布内 5）で較正の指標まで並べる。

| スコア | 定義 |
|---|---|
| 年齢のみ | 年齢 1 変数のロジスティック回帰（罰則なし）。各ユニットの訓練側で当て、test の年齢に適用 |
| 画像モデル | 融合後の p̂ = α / (α + β)。3 seed |
| 画像 + 年齢 | z + [logit π(a) − logit ȳ_train]。`auc_table` と同じ足し算で、学習し直したモデルではない |

どのスコアも目標施設のラベルは使わない。指標は AUC、ECE（10 等幅ビン、`openidh_model.eval.metrics` と同じ定義）、NLL、Brier、
平均予測、**較正切片**（傾き 1 のままロジットに足すべき量。正 = 過小評価）、**較正スロープ**（a + s·logit p の s。1 未満 = 過信）。

ロジックは `scripts/age_baseline_calibration.py`（この notebook はそれを import して使う）。
既存の結果ファイルは読むだけで、上書きしない。

In [ ]:
boot_reps = 1000                 # bootstrap reps for the age-only ECE interval
runs_dir  = "runs"
out_json  = "results/age_baseline_calibration.json"
out_md    = "results/age_baseline_calibration.md"
out_fig   = "results/figures/age_baseline_reliability.png"

In [ ]:
import os, sys, json
from pathlib import Path
_root = Path.cwd()
while not (_root / "data" / "metadata").exists():
    _root = _root.parent
os.chdir(_root); sys.path.insert(0, str(_root)); sys.path.insert(0, str(_root / "scripts"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from openidh_model.utils.paths import load_paths
import age_baseline_calibration as abc

for f in (out_json, out_md, out_fig):
    assert not Path(f).exists() or os.environ.get("AB_OVERWRITE") == "1", \
        f"{f} exists — refusing to overwrite an existing result (set AB_OVERWRITE=1 to rerun)"

paths = load_paths(os.environ.get("OPENIDH_PATHS", "configs/paths.local.yaml"))
print("paths:", os.environ.get("OPENIDH_PATHS", "configs/paths.local.yaml"), "| runs:", runs_dir)

## 0. 実行と検算

- **AUC の一致**：年齢のみ・画像モデルの AUC が `results/age_prior.json` の `auc_table` と一致すること（同じ分割・同じ当てはめ・同じ予測なので差は 0 のはず）
- **予測ファイル**：9 ユニット × 3 seed の `predictions.csv` がそろっていること

In [ ]:
rows = abc.evaluate(paths, Path(runs_dir), boot_reps)
summ = abc.summary(rows)
n_seeds = {r["label"]: len(r["seeds"]) for r in rows}
d_auc = abc.auc_crosscheck(rows)
checks = {"n_seeds": n_seeds, "max_abs_auc_diff_vs_age_prior": d_auc}
print("seeds per unit:", n_seeds)
print(f"max |ΔAUC| vs age_prior.json = {d_auc:.2e}")
assert all(v == 3 for v in n_seeds.values()), "predictions.csv が欠けているユニットがある"
assert d_auc is not None and d_auc < 1e-9

## 1. ユニットごとの較正

年齢の ECE の [ ] は test 症例のブートストラップ 95% 区間、画像の ± は seed 間 sd。

In [ ]:
recs = []
for r in rows:
    for sc in abc.SCORES:
        m = r[sc]
        recs.append({"unit": r["label"], "kind": r["kind"], "n": r["n_test"],
                     "prev_train": r["pi_train"], "prev_test": r["pi_test"], "score": sc,
                     **{k: m[k] for k in ("auc", "ece", "nll", "brier", "mean_p", "cal_intercept", "cal_slope")},
                     "ece_lo/sd": r["age_ece_ci95"][0] if sc == "age" else r[sc + "_sd"]["ece"],
                     "ece_hi": r["age_ece_ci95"][1] if sc == "age" else np.nan})
unit_df = pd.DataFrame(recs)
display(unit_df.round(3))

## 2. 条件ごとの平均（ユニット平均。分布外 = 4 ユニット、分布内 = 5 fold）

In [ ]:
summ_df = pd.DataFrame([{"kind": k, "score": sc, **m} for k, d in summ.items() for sc, m in d.items()])
display(summ_df.round(3))
wide = summ_df.pivot(index="score", columns="kind", values=["ece", "cal_intercept", "cal_slope", "auc"])
display(wide.round(3))

## 3. 信頼度図

分布外 4 ユニットと、分布内 5 fold をまとめたもの。対角線が完全較正。対角線より上は過小評価。
画像の曲線は 3 seed のビンを症例数で重み付けして合わせたもの。

In [ ]:
fig = abc.plot(rows, Path(out_fig))
plt.close(fig)
display(Image(filename=out_fig))

## 4. 読み方

結論はこのセルの上の数字から書く。見るべき点：

1. **分布外で、年齢のみの ECE は画像モデルより小さいか**。小さいなら「年齢は判別だけでなく較正もシフトに強い」と言える
2. **年齢のみの較正切片**：LOSO-B では既存の結果から +0.5 前後（予測平均 19.5% 対 実際 28.4%）が予想される。
   年齢で説明できない有病率差は、年齢モデルにも切片ずれとして残るはず
3. **較正スロープ**：画像モデルは 1 未満（過信）か。年齢モデルは 1 近くか
4. **画像 + 年齢**：AUC と同じく、ECE も両方の単独より良くなるか。それとも足し算で過信が強まるか（スロープ < 1）

## 5. 保存

In [ ]:
res = {"params": {"boot_reps": boot_reps, "runs_dir": runs_dir}, "checks": checks,
       "units": rows, "summary": summ}
Path(out_json).write_text(json.dumps(res, indent=2, ensure_ascii=False, default=float))
md_text = "# 年齢のみ・画像モデル・画像 + 年齢の較正 — 結果要約\n\n"
md_text += (f"検算：AUC は age_prior.json の auc_table と一致（最大差 {d_auc:.0e}）。"
            f"predictions.csv は全 9 ユニット × 3 seed。\n\n")
md_text += abc.tables_md(rows, summ) + "\n"
Path(out_md).write_text(md_text)
print("wrote", out_json, out_md, out_fig)